In [0]:
# Databricks notebook: process_quiz_data.py
# Reads raw Parquet files (landed by Azure Data Factory from MySQL on EC2),
# transforms them with pandas, and writes curated output back to ADLS Gen2.
#
# Requires: Unity Catalog External Locations already set up for the
# "raw" and "curated" containers (see project notes) - no storage keys
# needed here, access is handled via the Access Connector + Storage Credential.

# ---------------------------------------------------------------------------
# Cell 1: Paths
# ---------------------------------------------------------------------------
raw_path = "abfss://raw@stgquizdata.dfs.core.windows.net/mysql"
curated_path = "abfss://curated@stgquizdata.dfs.core.windows.net"

# ---------------------------------------------------------------------------
# Cell 2: Read raw tables (Spark reads from ADLS, then convert to pandas)
# ---------------------------------------------------------------------------
users_df = spark.read.parquet(f"{raw_path}/users").toPandas()
quizzes_df = spark.read.parquet(f"{raw_path}/quizzes").toPandas()
questions_df = spark.read.parquet(f"{raw_path}/questions").toPandas()
attempts_df = spark.read.parquet(f"{raw_path}/attempts").toPandas()
answers_df = spark.read.parquet(f"{raw_path}/answers").toPandas()

print("Row counts:")
print("users:", users_df.shape[0])
print("quizzes:", quizzes_df.shape[0])
print("questions:", questions_df.shape[0])
print("attempts:", attempts_df.shape[0])
print("answers:", answers_df.shape[0])

# ---------------------------------------------------------------------------
# Cell 3: Quiz performance - average score, attempt count, high/low per quiz
# ---------------------------------------------------------------------------
attempts_joined = (
    attempts_df
    .merge(users_df, left_on="user_id", right_on="id", suffixes=("", "_user"))
    .merge(quizzes_df, left_on="quiz_id", right_on="id", suffixes=("", "_quiz"))
)

attempts_joined = attempts_joined[["id", "name", "title", "score", "submitted_at"]]
attempts_joined.columns = ["attempt_id", "student_name", "quiz_title", "score", "submitted_at"]

# Drop attempts that were started but never submitted (score is null)
completed = attempts_joined.dropna(subset=["score"])

quiz_performance = completed.groupby("quiz_title").agg(
    avg_score=("score", "mean"),
    total_attempts=("attempt_id", "count"),
    highest_score=("score", "max"),
    lowest_score=("score", "min"),
).reset_index()

print(quiz_performance)

# ---------------------------------------------------------------------------
# Cell 4: Question difficulty - accuracy per question
# ---------------------------------------------------------------------------
answers_joined = answers_df.merge(
    questions_df, left_on="question_id", right_on="id", suffixes=("", "_q")
)
answers_joined = answers_joined[["question_text", "is_correct"]]

question_difficulty = answers_joined.groupby("question_text").agg(
    total_answered=("is_correct", "count"),
    correct_count=("is_correct", "sum"),
).reset_index()

question_difficulty["accuracy_pct"] = round(
    (question_difficulty["correct_count"] / question_difficulty["total_answered"]) * 100, 2
)

print(question_difficulty)

# ---------------------------------------------------------------------------
# Cell 5: Write curated output back to ADLS Gen2 (overwrite each run)
# ---------------------------------------------------------------------------
spark.createDataFrame(quiz_performance).write.mode("overwrite").parquet(
    f"{curated_path}/quiz_performance"
)
spark.createDataFrame(question_difficulty).write.mode("overwrite").parquet(
    f"{curated_path}/question_difficulty"
)

print("Curated data written successfully")

Row counts:
users: 4
quizzes: 7
questions: 70
attempts: 6
answers: 40
                   quiz_title  avg_score  ...  highest_score  lowest_score
0    Java Interview Questions        8.0  ...            8.0           8.0
1  Python Interview Questions        9.5  ...           10.0           9.0
2         interview Questions        6.0  ...            6.0           6.0

[3 rows x 5 columns]
                                        question_text  ...  accuracy_pct
0   1. Which keyword is used to define a function ...  ...         100.0
1   A bag contains 5 red, 4 blue, and 3 green ball...  ...           0.0
2   A number is increased by 25% and then decrease...  ...         100.0
3   A person walks 12 km north, then 5 km east. Ho...  ...           0.0
4   A shopkeeper marks an item 40% above the cost ...  ...           0.0
5   A sum of money becomes ₹12,000 at 20% simple i...  ...           0.0
6   A train 180 m long crosses a pole in 12 second...  ...         100.0
7   Find the next number

In [0]:
# Overall summary metrics for the dashboard
summary_metrics = {
    "total_users": [users_df[users_df["role"] == "student"].shape[0]],
    "total_quizzes": [quizzes_df.shape[0]],
    "total_questions": [questions_df.shape[0]],
    "total_attempts": [attempts_df.shape[0]],
    "completed_attempts": [attempts_df["score"].notna().sum()],
    "abandoned_attempts": [attempts_df["score"].isna().sum()],
    "overall_avg_score": [round(attempts_df["score"].mean(), 2)],
}

import pandas as pd
summary_df = pd.DataFrame(summary_metrics)
print(summary_df)

   total_users  total_quizzes  ...  abandoned_attempts  overall_avg_score
0            3              7  ...                   2               8.25

[1 rows x 7 columns]


In [0]:
spark.createDataFrame(summary_df).write.mode("overwrite").parquet(f"{curated_path}/summary_metrics")
print("Summary metrics written successfully")

Summary metrics written successfully
